# Iceberg 1.12: streaming writes and Parquet tuning

Two practical 1.12 additions for Spark (3.5, 4.0 and 4.1):

1. **`use-merge-append-for-streaming`**: a streaming-write option. Structured Streaming commits every micro-batch; by default each commit adds a *new small manifest file*,
   so a long-running stream piles up manifests and slows down query planning. With the option, manifests are merged on commit.
2. **Per-column Parquet dictionary encoding** (`write.parquet.dict-encoding-enabled.column.<col>`): switch dictionary encoding off for individual columns.

Both are checked below against the lab's own tables and files.

In [ ]:
import sys, pathlib, time
sys.path.insert(0, str(pathlib.Path.cwd().parent))
from lab_utils import get_spark, parquet_footer, CATALOG

spark = get_spark("iceberg-streaming")
print("Spark", spark.version)
NS = f"{CATALOG}.iceberg12"
_ = spark.sql(f"CREATE NAMESPACE IF NOT EXISTS {NS}")

## 1. Streaming into Iceberg: manifests per commit
We stream the built-in `rate` source (20 rows per second) into two identical tables with 1-second micro-batches for 30 seconds each, once with the default
(fast append) and once with `use-merge-append-for-streaming = true`.

Manifests are only merged when there are at least `commit.manifest.min-count-to-merge` of them (**default 100**). A 30-second demo would never reach that, so both tables
set it to **10** to make the effect visible. On a real stream, the default threshold applies.

In [ ]:
def run_stream(name, merge_append, seconds=30):
    spark.sql(f"DROP TABLE IF EXISTS {NS}.{name}")
    spark.sql(f"""CREATE TABLE {NS}.{name} (timestamp TIMESTAMP, value BIGINT) USING iceberg
                  TBLPROPERTIES ('commit.manifest.min-count-to-merge' = '10')""")
    query = (spark.readStream.format("rate").option("rowsPerSecond", 20).load()
             .writeStream.format("iceberg").outputMode("append")
             .trigger(processingTime="1 second")
             .option("checkpointLocation", f"s3a://warehouse/checkpoints/{name}_{int(time.time())}")
             .option("use-merge-append-for-streaming", str(merge_append).lower())
             .toTable(f"{NS}.{name}"))
    time.sleep(seconds)
    query.stop()

def stats(name):
    count = lambda meta: spark.sql(f"SELECT count(*) FROM {NS}.{name}.{meta}").collect()[0][0]
    return {"snapshots": count("snapshots"), "manifests": count("manifests"), "data files": count("files")}

run_stream("stream_fast_append", merge_append=False)
run_stream("stream_merge_append", merge_append=True)
import pandas as pd
pd.DataFrame({"fast append (default)": stats("stream_fast_append"), "merge append": stats("stream_merge_append")}).T

Same number of commits and data files, but far fewer manifests with merge-append. (Run it again and the exact numbers shift a little with timing.)

The classic fix for an already-bloated table is the `rewrite_manifests` procedure. It works on the fast-append table too:

In [ ]:
before = stats("stream_fast_append")["manifests"]
spark.sql(f"CALL {CATALOG}.system.rewrite_manifests('iceberg12.stream_fast_append')").show(truncate=False)
print("manifests:", before, "->", stats("stream_fast_append")["manifests"])

Prefer the write option for streams that run for days; schedule `rewrite_manifests` as maintenance for tables that already have the problem.
(Iceberg 1.12 also added a `sort_by` parameter to `rewrite_manifests` to cluster manifests by partition transforms; the release notes list Spark 3.5 and 4.0, and it ran fine on Spark 4.1.3 here: see notebook 00.)

## 2. Parquet: dictionary encoding per column
Dictionary encoding helps low-cardinality columns and wastes effort on columns where almost every value is unique. In 1.12 you can control it for a single column.
We write the same rows twice and read the Parquet footer of the data file.

In [ ]:
cases = {"parquet_default": "", "parquet_no_dict_code": ", 'write.parquet.dict-encoding-enabled.column.code' = 'false'"}
for name, props in cases.items():
    spark.sql(f"DROP TABLE IF EXISTS {NS}.{name}")
    spark.sql(f"CREATE TABLE {NS}.{name} (id BIGINT, code STRING, amount DOUBLE) USING iceberg TBLPROPERTIES ('format-version' = '2'{props})")
    spark.sql(f"INSERT INTO {NS}.{name} SELECT id, concat('C', id % 500), rand(5) FROM range(0, 100000)")
    path = spark.sql(f"SELECT file_path FROM {NS}.{name}.files LIMIT 1").collect()[0][0]
    size = spark.sql(f"SELECT sum(file_size_in_bytes) FROM {NS}.{name}.files").collect()[0][0]
    col = parquet_footer(path).row_group(0).column(1)          # column 1 = `code`
    print(f"{name:22s} {size:>8,} bytes | code encodings: {sorted(col.encodings)} | dictionary page: {col.has_dictionary_page}")

With the property, the `code` column loses its dictionary page and is stored with `PLAIN` encoding (the default table keeps `PLAIN_DICTIONARY`). Whether that makes the file larger or smaller depends on the data,
so check with your own columns.

## Notes and limits
* Table properties for Parquet writes are listed in the Iceberg [configuration reference](https://iceberg.apache.org/docs/latest/configuration/#write-properties).
* 1.12 also documents **adaptive bloom-filter sizing** (`write.parquet.bloom-filter-adaptive-enabled`). In the tests behind these notebooks no bloom filter could be found in the Parquet files written
  through this stack (checked with the footer metadata of pyarrow and DuckDB), so bloom filters are deliberately **not** demonstrated here.
* The streaming checkpoints are written to `s3a://warehouse/checkpoints/` in SeaweedFS.

In [ ]:
CLEANUP = False   # set True to drop the tables created here
if CLEANUP:
    for t in ["stream_fast_append", "stream_merge_append", "parquet_default", "parquet_no_dict_code"]:
        spark.sql(f"DROP TABLE IF EXISTS {NS}.{t}")